In [7]:
import os
import json
from statistics import mean, median
from transformers import AutoTokenizer

# ---- CONFIG ----
FOLDER_PATH = "/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/privacy_qa"   # CHANGE THIS
OUTPUT_JSON = "/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/meta-enriched-rag-for-legal-llms/RAG_pipeline/Token_level_data/privacy_qa.json"
MODEL_NAME = "meta-llama/Llama-3.2-3B"   # <-- EXACT TOKENIZER FOR 3.2-3B

# ---- LOAD TOKENIZER ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ---- PROCESS FILES ----
token_counts = {}

for file_name in os.listdir(FOLDER_PATH):
    if file_name.endswith(".txt"):
        file_path = os.path.join(FOLDER_PATH, file_name)
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        token_count = len(tokenizer.encode(text))
        token_counts[file_name] = token_count
        print(f"{file_name}: {token_count} tokens")

# ---- SAVE RESULTS ----
with open(OUTPUT_JSON, "w") as f:
    json.dump(token_counts, f, indent=4)

# ---- STATISTICS ----
values = list(token_counts.values())

print("\n==== STATISTICS ====")
print(f"Files processed: {len(values)}")
print(f"Total tokens: {sum(values)}")
print(f"Average tokens: {mean(values):.2f}")
print(f"Median tokens: {median(values):.2f}")
print(f"Min tokens: {min(values)} ({min(token_counts, key=token_counts.get)})")
print(f"Max tokens: {max(values)} ({max(token_counts, key=token_counts.get)})")

print(f"\nSaved token stats to {OUTPUT_JSON}")


23andMe.txt: 9969 tokens
Fiverr.txt: 4791 tokens
Groupon.txt: 5087 tokens
Keep.txt: 2025 tokens
TickTick: To Do List with Reminder, Day Planner.txt: 594 tokens
Viber Messenger.txt: 5406 tokens
Wordscapes.txt: 5434 tokens

==== STATISTICS ====
Files processed: 7
Total tokens: 33306
Average tokens: 4758.00
Median tokens: 5087.00
Min tokens: 594 (TickTick: To Do List with Reminder, Day Planner.txt)
Max tokens: 9969 (23andMe.txt)

Saved token stats to /home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/meta-enriched-rag-for-legal-llms/RAG_pipeline/Token_level_data/privacy_qa.json


In [1]:
import os
from transformers import AutoTokenizer


def chunk_folder_llama(
    input_folder: str,
    output_folder: str,
    chunk_size: int = 500,
    window: int = 50,
    model_name: str = "thenlper/gte-large"
):
    # Load tokenizer once
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    folder_name = os.path.basename(os.path.normpath(input_folder))

    for filename in os.listdir(input_folder):
        if not filename.lower().endswith(".txt"):
            continue

        filepath = os.path.join(input_folder, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        # Tokenize full file
        tokens = tokenizer.encode(text)
        n = len(tokens)

        start = 0
        chunk_idx = 1

        while start < n:
            end = start + chunk_size
            chunk_tokens = tokens[start:end]

            # Decode back to text
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)

            # Output file naming
            out_name = f"{folder_name}_{filename}_chunk{chunk_idx}.txt"
            out_path = os.path.join(output_folder, out_name)

            with open(out_path, "w", encoding="utf-8") as out_f:
                out_f.write(chunk_text)

            chunk_idx += 1

            # Sliding window step
            start += chunk_size - window

chunk_folder_llama(
    input_folder="/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/privacy_qa",
    output_folder="/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/Chunks_data/privacy_qa_500_encoder",
    chunk_size=500,
    window=50,
    model_name="thenlper/gte-large"
)


/work/pi_hongyu_umass_edu/smaniyar_umass_edu-conda/envs/llamafactoryenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os

def fix_chunk_filenames(input_folder: str):
    for filename in os.listdir(input_folder):
        if not filename.endswith(".txt"):
            continue

        # Example: privacy_qa_23andMe.txt_chunk1.txt
        # We want to remove the .txt before "_chunk"
        if ".txt_chunk" in filename:
            new_name = filename.replace(".txt_chunk", "_chunk")
            old_path = os.path.join(input_folder, filename)
            new_path = os.path.join(input_folder, new_name)
            os.rename(old_path, new_path)
            # print(f"Renamed: {filename}  ->  {new_name}")


In [5]:
fix_chunk_filenames('/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/Chunks_data/privacy_qa_500_encoder')

In [11]:
%cd /home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/meta-enriched-rag-for-legal-llms/RAG_pipeline

/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/meta-enriched-rag-for-legal-llms/RAG_pipeline


In [6]:
import os
import json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import faiss
from glob import glob

# =========================
# CONFIG
# =========================
EMBED_MODEL = "thenlper/gte-large"     # or any open-source embedding model
CHUNK_FOLDER = "/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/Chunks_data/privacy_qa_500_encoder"            # folder where your chunk files live
FAISS_INDEX_PATH = "faiss_index_privacy_qa.bin"   # faiss file
META_PATH = "metadata.json"            # id → metadata mapping
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 16
NORMALIZE = True

EMBED_MODEL = "thenlper/gte-large"     
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL, use_fast=True)
model = AutoModel.from_pretrained(EMBED_MODEL).to(DEVICE)
model.eval()


# =========================
# MEAN POOLING
# =========================
def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden * mask).sum(dim=1)
    count = mask.sum(dim=1).clamp(min=1e-9)
    return summed / count


# =========================
# ENCODER
# =========================
def encode_texts(text_list):
    out = []
    for i in range(0, len(text_list), BATCH):
        b = text_list[i:i+BATCH]
        inputs = tokenizer(b, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            output = model(**inputs, return_dict=True)
            emb = mean_pool(output.last_hidden_state, inputs["attention_mask"])
        emb = emb.cpu().numpy().astype("float32")
        out.append(emb)

    embs = np.vstack(out)
    if NORMALIZE:
        faiss.normalize_L2(embs)
    return embs


# =========================
# BUILD INDEX
# =========================
def build_faiss_index():
    chunk_files = sorted(glob(os.path.join(CHUNK_FOLDER, "*.txt")))
    print("Found", len(chunk_files), "chunk files.")

    # get embedding dim
    test_vec = encode_texts(["hello world"])
    dim = test_vec.shape[1]

    index = faiss.IndexFlatIP(dim)      # cosine via inner-product
    index = faiss.IndexIDMap(index)

    metadata = {}
    next_id = 1

    for fpath in chunk_files:
        with open(fpath, "r", encoding="utf-8") as f:
            text = f.read()

        emb = encode_texts([text])
        id_arr = np.array([next_id], dtype=np.int64)

        index.add_with_ids(emb, id_arr)

        metadata[str(next_id)] = {
            "file": fpath
        }

        next_id += 1

    faiss.write_index(index, FAISS_INDEX_PATH)
    with open(META_PATH, "w", encoding="utf-8") as mf:
        json.dump(metadata, mf, indent=2)

    print("Index saved:", FAISS_INDEX_PATH)
    print("Metadata saved:", META_PATH)


# =========================
# SEARCH PIPELINE
# =========================
def load_search_components():
    index = faiss.read_index(FAISS_INDEX_PATH)
    with open(META_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    return index, metadata


def search(query, topk=5):
    index, metadata = load_search_components()

    q_emb = encode_texts([query])
    scores, ids = index.search(q_emb, topk)

    results = []
    for s, i in zip(scores[0], ids[0]):
        if i == -1:
            continue
        meta = metadata[str(int(i))]
        results.append({
            "id": int(i),
            "score": float(s),
            "file": meta["file"]
        })
    return results


# =========================
# MAIN (optional examples)
# =========================



: 

: 

In [5]:
build_faiss_index()


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Found 79 chunk files.


RuntimeError: The size of tensor a (516) must match the size of tensor b (512) at non-singleton dimension 1

In [ ]:
#### JSON approach 

In [2]:
import os
import json
from transformers import AutoTokenizer


def chunk_folder_llama_json(
    input_folder: str,
    output_json: str,
    chunk_size: int = 500,
    window: int = 50,
    model_name: str = "thenlper/gte-large"
):
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    # Final dictionary to save
    result = {}

    # Identify folder name for prefix
    folder_name = os.path.basename(os.path.normpath(input_folder))

    # Iterate over all .txt files
    for filename in os.listdir(input_folder):
        if not filename.lower().endswith(".txt"):
            continue

        filepath = os.path.join(input_folder, filename)

        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        # Remove extension for chunk_id
        base_name = os.path.splitext(filename)[0]

        # Tokenize with offsets
        encoded = tokenizer(
            text,
            return_offsets_mapping=True,
            add_special_tokens=False
        )

        token_ids = encoded["input_ids"]
        offsets = encoded["offset_mapping"]
        n = len(token_ids)

        start = 0
        chunk_idx = 1

        # Sliding window chunking
        while start < n:
            end = min(start + chunk_size, n)

            chunk_offsets = offsets[start:end]

            # Exact character span in original raw text
            char_start = chunk_offsets[0][0]
            char_end = chunk_offsets[-1][1]

            chunk_text = text[char_start:char_end]

            # Construct chunk ID (no .txt)
            chunk_id = f"{folder_name}_{base_name}_chunk{chunk_idx}"

            # Save into dictionary
            result[chunk_id] = {
                "chunk_text": chunk_text,
                "span": [char_start, char_end]
            }

            chunk_idx += 1
            start += chunk_size - window

    # Write entire dictionary as one JSON
    with open(output_json, "w", encoding="utf-8") as jf:
        json.dump(result, jf, ensure_ascii=False, indent=2)




/work/pi_hongyu_umass_edu/smaniyar_umass_edu-conda/envs/llamafactoryenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
chunk_folder_llama_json(
    input_folder="/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/privacy_qa",
    output_json="/home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/privacy_qa_chunks.json",
    chunk_size=500,
    window=50,
    model_name="thenlper/gte-large"
)


In [4]:
with open("home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/privacy_qa_chunks.json") as w:
    a=json.load(w)

FileNotFoundError: [Errno 2] No such file or directory: 'home/smaniyar_umass_edu/BioNLP_Ontology/other/nlp/RAG_data/privacy_qa_chunks.json'